In [ ]:
import pandas as pd
from datasets import Dataset

In [ ]:
data_path = '/Users/jk/stroke_datasets/stroke_registry_post_hoc_modified.xlsx'

In [ ]:
df = pd.read_excel(data_path)

In [ ]:
train_df = df[['Age (calc.)', 'NIH on admission', '3M Death']]
train_dataset = Dataset.from_pandas(train_df)

In [ ]:
train_dataset

In [ ]:
train_df['NIH on admission'].dtype

In [ ]:
def create_registry_case_identification_column(df):
    # Identify each case with case id (patient id + eds last 4 digits)
    df = df.copy()
    if 'patient_id' not in df.columns:
        df['patient_id'] = df['Case ID'].apply(lambda x: x[8:-4]).astype(str)
    if 'EDS_last_4_digits' not in df.columns:
        df['EDS_last_4_digits'] = df['Case ID'].apply(lambda x: x[-4:]).astype(str)
    case_identification_column = df['patient_id'].astype(str) \
                                 + '_' + df['EDS_last_4_digits'].str.zfill(4).astype(str)
    return case_identification_column

In [ ]:
df['case_admission_id'] = create_registry_case_identification_column(df)

In [ ]:
outcome = '3M Death'

In [ ]:
data = df.copy()
data['patient_id'] = data['case_admission_id'].apply(lambda x: x.split('_')[0])

"""
SPLITTING DATA
Splitting is done by patient id (and not admission id) as in case of the rare multiple admissions per patient there
would be a risk of data leakage otherwise split 'pid' in TRAIN and TEST pid = unique patient_id
"""
# Reduce every patient to a single outcome (to avoid duplicates)
all_pids = data.patient_id.unique()
all_outcomes = data[data.patient_id.isin(all_pids)]
# keep only maximum outcome per patient (in case of multiple admissions per patient)
all_outcomes = all_outcomes.groupby('patient_id')[outcome].max().reset_index()
# let all outcomes be an array of outcomes corresponding to the unique patient ids
all_outcomes = all_outcomes[outcome].values


In [ ]:
pid_train, pid_test, y_pid_train, y_pid_test = train_test_split(all_pids,
                                                                    all_outcomes,
                                                                    stratify=all_outcomes,
                                                                    test_size=test_size,
                                                                    random_state=seed)

train_set = data[data.patient_id.isin(pid_train)]        
test_set = data[data.patient_id.isin(pid_test)]
num_train = len(train_set)
num_test = len(test_set)